In [ ]:
!pip install pymupdf4llm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 28.0 MB/s eta 0:00:00


In [ ]:
import pymupdf4llm
import glob
import pathlib

# This finds all your uploaded PDFs automatically
pdf_files = glob.glob("*.pdf")
full_context = ""

print(f"Found {len(pdf_files)} files. Starting extraction...")

for pdf in pdf_files:
    print(f"Extracting: {pdf}")
    # Convert PDF to Markdown to keep tables and structure intact
    md_text = pymupdf4llm.to_markdown(pdf)
    full_context += f"\n\n--- Source: {pdf} ---\n\n" + md_text

# Save the combined text for the next step
pathlib.Path("combined_circular_economy_data.txt").write_bytes(full_context.encode())

print("\nSuccess! All text extracted and saved to 'combined_circular_economy_data.txt'.")
print(f"Total characters extracted: {len(full_context)}")

Found 3 files. Starting extraction...
Extracting: Enhancing-Circular-Economy-of-End-of-Life-Vehicles-ELVs-in-India.pdf
Extracting: Enhancing-Circular-Economy-of-Waste-Tyres-in-India.pdf
Extracting: Advancing-Circular-Economy-of-Waste-Electronic-and-Electrical-Equipment-Ewaste-and-Lithium-Ion-Batteries-in-India.pdf

Success! All text extracted and saved to 'combined_circular_economy_data.txt'.
Total characters extracted: 329052


In [ ]:
!pip install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.4/724.4 kB 55.6 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.64.0
    Uninstalling google-genai-1.64.0:
      Successfully uninstalled google-genai-1.64.0


In [ ]:
!pip install huggingface_hub requests


In [ ]:
import os
import json
import time
import re
from huggingface_hub import InferenceClient
from google.colab import userdata, files

# 1. Setup Client
client = InferenceClient(api_key=userdata.get('HF_TOKEN'))

# 2. Files
text_file = "combined_circular_economy_data.txt"
output_file = "circular_economy_dataset.jsonl"

def generate_robust_dataset(text_file, output_file):
    with open(text_file, 'r') as f:
        content = f.read()

    chunks = [content[i:i+8000] for i in range(0, len(content), 8000)]

    # RESUME LOGIC: Detects existing lines and starts exactly where you left off
    start_chunk = 0
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            lines = f.readlines()
            start_chunk = len(lines) // 5
        print(f"🔄 Resuming from chunk {start_chunk + 1}...")

    for i in range(start_chunk, len(chunks)):
        chunk = chunks[i]
        prompt = f"Return ONLY a JSON list of 5 Instruction-Output pairs from this text: {chunk}"

        success = False
        while not success: # FIX: Infinite retry ensures no chunk is ever skipped
            try:
                response = client.chat_completion(
                    model="meta-llama/Llama-3.3-70B-Instruct",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=1500
                )

                raw_text = response.choices[0].message.content

                # THE FIX: Use Regex to find the JSON list even if there is 'Extra data'
                match = re.search(r'\[.*\]', raw_text, re.DOTALL)
                if match:
                    clean_json = match.group(0)
                    pairs = json.loads(clean_json) #

                    with open(output_file, 'a') as f:
                        for entry in pairs:
                            f.write(json.dumps(entry) + '\n')

                    print(f"✅ Chunk {i+1}/{len(chunks)} saved successfully.")
                    success = True # Move to the next chunk
                    time.sleep(12) # Safety delay for free tier
                else:
                    raise ValueError("AI failed to output a valid JSON list format.")

            except Exception as e:
                # Handle 503 (Busy) and 429 (Rate Limit) by waiting and retrying
                print(f"⏳ Issue at {i+1}: {e}. Retrying same chunk in 60s...")
                time.sleep(60)

generate_robust_dataset(text_file, output_file)

🔄 Resuming from chunk 24...
✅ Chunk 24/42 saved successfully.
✅ Chunk 25/42 saved successfully.
✅ Chunk 26/42 saved successfully.
✅ Chunk 27/42 saved successfully.
✅ Chunk 28/42 saved successfully.
✅ Chunk 29/42 saved successfully.
✅ Chunk 30/42 saved successfully.
✅ Chunk 31/42 saved successfully.
✅ Chunk 32/42 saved successfully.
✅ Chunk 33/42 saved successfully.
✅ Chunk 34/42 saved successfully.
✅ Chunk 35/42 saved successfully.
✅ Chunk 36/42 saved successfully.
✅ Chunk 37/42 saved successfully.
✅ Chunk 38/42 saved successfully.
✅ Chunk 39/42 saved successfully.
✅ Chunk 40/42 saved successfully.
✅ Chunk 41/42 saved successfully.
✅ Chunk 42/42 saved successfully.


In [ ]:
with open("circular_economy_dataset.jsonl", "r") as f:
    total_lines = len(f.readlines())
print(f"Total pairs generated: {total_lines}")

Total pairs generated: 212


In [ ]:
import json

# Alpaca prompt template for Unsloth
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

formatted_data = []
skipped_lines = 0

with open("circular_economy_dataset.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if not line: continue # Skip empty lines

        try:
            data = json.loads(line)
            # Check if it's a dictionary before accessing keys
            if isinstance(data, dict) and "instruction" in data and "output" in data:
                formatted_data.append({
                    "instruction": data["instruction"],
                    "output": data["output"],
                    "text": alpaca_prompt.format(data["instruction"], data["output"])
                })
            else:
                skipped_lines += 1
        except Exception:
            skipped_lines += 1

# Save as a single JSON file for Unsloth
with open("unsloth_dataset.json", "w") as f:
    json.dump(formatted_data, f, indent=2)

print(f"✅ Successfully formatted {len(formatted_data)} examples!")
if skipped_lines > 0:
    print(f"⚠️ Skipped {skipped_lines} malformed lines.")

✅ Successfully formatted 120 examples!
⚠️ Skipped 92 malformed lines.


In [ ]:
import json
import re

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

recovered_data = []
with open("circular_economy_dataset.jsonl", "r") as f:
    for line in f:
        # THE FIX: Find everything between [ ] or { } even if the line is 'dirty'
        matches = re.findall(r'\{.*?"instruction":.*?"output":.*?\}', line)

        for match in matches:
            try:
                data = json.loads(match)
                recovered_data.append({
                    "instruction": data["instruction"],
                    "output": data["output"],
                    "text": alpaca_prompt.format(data["instruction"], data["output"])
                })
            except:
                continue

with open("unsloth_dataset.json", "w") as f:
    json.dump(recovered_data, f, indent=2)

print(f"🎯 Total Examples Recovered: {len(recovered_data)}")

🎯 Total Examples Recovered: 115


In [ ]:
# 1. Force-upgrade the specific conflicting package
!pip install --no-deps "torchao>=0.13.0"

# 2. Fix the minor version drift in fsspec/gcsfs (Optional but stops the red text)
!pip install "fsspec==2025.3.0" "gcsfs==2025.3.0"

# 3. Verify that the Unsloth engine can now find its dependencies
import torchao
print(f"✅ TorchAO version: {torchao.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.12.0
    Uninstalling fsspec-2024.12.0:
      Successfully uninstalled fsspec-2024.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.4.1 requires fsspec[http]<=2024.12.0,>=2023.1.0, but you have fsspec 2025.3.0 which is incompatible.
✅ TorchAO version: 0.16.0


In [ ]:
import json

# Load the formatted dataset to check for any leftover malformations
with open("unsloth_dataset.json", "r") as f:
    final_data = json.load(f)

print(f"📊 Final Dataset Size: {len(final_data)} examples")
print(f"✅ Sample Instruction: {final_data[0]['instruction'][:100]}...")

📊 Final Dataset Size: 115 examples
✅ Sample Instruction: What is the primary focus of the report "Enhancing Circular Economy of End-of-Life Vehicles (ELVs) i...


In [ ]:
# Remove everything
!pip uninstall -y unsloth unsloth_zoo trl peft accelerate bitsandbytes transformers

# Install compatible 2026 stack
!pip install --upgrade --no-cache-dir \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" \
    trl==0.24.0 \
    transformers==4.56.2 \
    accelerate \
    peft \
    bitsandbytes

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-l6nnk0xv/unsloth_c261c20d97bc4634bc4137cea8c55da2
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-l6nnk0xv/unsloth_c261c20d97bc4634bc4137cea8c55da2
  Resolved https://github.com/unslothai/unsloth.git to commit 98c558364d17f57e363c170ef2bc8f57cd1c33d9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-install-l6nnk0xv/unsloth-zoo_97b9c40b01b54c0ab3e7de73e

In [ ]:
from unsloth import FastLanguageModel
print("✅ Unsloth loaded successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth loaded successfully


In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# 1. Load the optimized Gemma 3 model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Updated LoRA with higher capacity
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Increased rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # All projections
    lora_alpha = 32, # Matches rank
    lora_dropout = 0,
    bias = "none",
)

# 3. Load your formatted dataset
dataset = load_dataset("json", data_files="unsloth_dataset.json", split="train")


# Updated Training Config for deeper learning
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 150, # More steps to lower the loss
        learning_rate = 1e-4, # Slightly lower LR for stability at higher steps
        fp16 = not torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        weight_decay = 0.01, # Helps prevent overfitting during extra steps
        output_dir = "circular_economy_v2",
    ),
)
# 3. Load your formatted dataset
dataset = load_dataset("json", data_files="unsloth_dataset.json", split="train")



trainer.train()

==((====))==  Unsloth 2026.2.1: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients
Unsloth: Switching to float32 training since model cannot work with float16


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 115 | Num Epochs = 10 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 65,576,960 of 4,365,656,432 (1.50% trained)


Step,Training Loss
1,15.708300
2,13.408700
3,16.063400
4,14.617600
5,14.857800
6,13.772700
7,13.017900
8,11.113300
9,11.677500
10,11.246300


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=150, training_loss=3.295069428086281, metrics={'train_runtime': 1267.099, 'train_samples_per_second': 0.947, 'train_steps_per_second': 0.118, 'total_flos': 2981450255148000.0, 'train_loss': 3.295069428086281, 'epoch': 10.0})

In [ ]:
from unsloth import FastLanguageModel

# 1. Use the model and tokenizer already in your memory
FastLanguageModel.for_inference(model) # Enable 2x faster inference

# 2. Define the project-specific instruction
instruction = "What are the specific NITI Aayog recommendations for the 'circularity' of End-of-Life Vehicles (ELVs) in India?"
input_context = ""

# 3. Format the prompt using the Alpaca style used in your training
prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

# 4. Use the existing 'tokenizer' variable from your training cell
inputs = tokenizer(
    [prompt.format(instruction, "")],
    return_tensors = "pt"
).to("cuda")

# 5. Generate and decode the output
outputs = model.generate(**inputs, max_new_tokens = 512, temperature = 0.3)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What are the specific NITI Aayog recommendations for the 'circularity' of End-of-Life Vehicles (ELVs) in India?

### Response:
NITI Aayog recommends strengthening the regulatory framework for ELV management to classify RVSFs as industrial or commercial establishments, introduce an extended producer responsibility (EPR) mechanism for scrap tyre management, and establish a separate stream for ELVs from the informal sector. It further suggests revising EPR targets for tyre recycling to 50% and increasing the EPR target for scrap rubber to 75% to be revised every five years based on actual production and consumption figures. Additionally, the Governing Board for ELV management should be reconstituted with representatives from MoRTH, OEMs, RVSFs, and state authorities, and the committee should be mandated to revise EP

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

instruction = "What are the specific NITI Aayog recommendations for the 'circularity' of End-of-Life Vehicles (ELVs) in India?"
prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

inputs = tokenizer([prompt.format(instruction, "")], return_tensors = "pt").to("cuda")

# THE FIX: Added repetition_penalty and changed temperature/top_p
outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    temperature = 0.7,      # Higher temperature = more creativity
    top_p = 0.9,            # Only considers the top 90% most likely words
    repetition_penalty = 1.2 # Strong penalty to stop the looping
)

print(tokenizer.decode(outputs[0], skip_special_tokens = True))

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What are the specific NITI Aayog recommendations for the 'circularity' of End-of-Life Vehicles (ELVs) in India?

### Response:
NITI Aayog recommends strengthening EPR targets to include future growth and incorporating all ELV components under EPR; revising conversion factors to better reflect actual material recovery rates; and separating battery chemistries from core scrap data for accurate EPR accounting. It also suggests extending the validity period of existing batch normalization rules by five years and clarifying Extended Producer Responsibility (EPR) obligations for spare parts. Furthermore, it proposes integrating digital technologies like Vehicle Scrapping Facilitation Platforms across states and standardizing scrap valuation methodologies using standardized templates based on current metal prices dissem

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a specific folder for this version
save_directory = "/content/drive/MyDrive/NITI_Aayog_Gemma3_v2"

model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"✅ Saved successfully to: {save_directory}")

Mounted at /content/drive
✅ Saved successfully to: /content/drive/MyDrive/NITI_Aayog_Gemma3_v2
